# Øvingsoppgaver 2 – Adversarial attacks
Data Mining & Analytics – Samling 3, dag 1


Angrepene er ferdig implementert i koden under. Oppgavene handler primært om  om å bruke funksjonene og tolke resultatene. Hver oppgave har startkode der du fyller inn det som er merket `TODO`.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.5)
rng = np.random.default_rng(42)
tf.keras.utils.set_random_seed(42)
print("TensorFlow", tf.__version__)

In [ ]:
klasser = ["T-skjorte/topp", "Bukse", "Genser", "Kjole", "Kåpe",
           "Sandal", "Skjorte", "Joggesko", "Veske", "Ankelstøvel"]

In [ ]:
# Data og to modeller (samme arkitekturer som i øvelse 5) – kjør denne cellen først
(X_full, y_full), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
X_full = (X_full / 255.0).astype("float32")[..., np.newaxis]
X_test = (X_test / 255.0).astype("float32")[..., np.newaxis]
X_train, y_train = X_full[:20_000], y_full[:20_000]
X_ev, y_ev = X_test[:2_000], y_test[:2_000]              # evalueringsutvalg

def lag_mlp():
    return tf.keras.Sequential([
        tf.keras.Input(shape=(28, 28, 1)), tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation="relu"), tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation="relu"), tf.keras.layers.Dense(10, activation="softmax")], name="MLP")

def lag_cnn(navn="CNN"):
    return tf.keras.Sequential([
        tf.keras.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"), tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"), tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Flatten(), tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation="relu"), tf.keras.layers.Dense(10, activation="softmax")], name=navn)

mlp, cnn = lag_mlp(), lag_cnn()
for m in (mlp, cnn):
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    m.fit(X_train, y_train, epochs=6, batch_size=64, verbose=0)
    print(f"{m.name}: testnøyaktighet {m.evaluate(X_test, y_test, verbose=0)[1]:.3f}")

def evaluer(modell, x, y):
    """Nøyaktighet og gjennomsnittlig konfidens på feilklassifiserte eksempler."""
    p = modell.predict(x, verbose=0, batch_size=500)
    feil = p.argmax(axis=1) != y
    return {"nøyaktighet": 1 - feil.mean(), "konfidens på feil": p.max(axis=1)[feil].mean() if feil.any() else np.nan}

## Forhåndsdefinerte funksjoner

| Funksjon | Hva den gjør |
|---|---|
| `fgsm(modell, x, y, eps)` | Legger til støy i bildet som er optimalisert for å minimere aktiveringen av en hvis klasse |
| `fgsm_malrettet(modell, x, maal, eps)` | Målrettet FGSM legger til støy i bildet som er optimalisert for å maksimere aktiveringen av en gitt klasse  |
| `tilfeldig_stoy(x, eps)` | Tilfeldig støy per piksel, samme størrelse som FGSM, men tilfeldig retning |
| `vis_angrep(modell, x, x_adv)` | Viser originalbilde, forstyrrelse og adversarial bilde med prediksjon og konfidens |
| `adversarial_trening(modell, X, y, eps, alfa, epoker)` | Trener en modell med adversarial-eksempler |


In [ ]:
loss_func = tf.keras.losses.SparseCategoricalCrossentropy()

def fgsm(modell, x, y, eps):
    """Ikke-målrettet FGSM (Goodfellow et al. 2015)."""
    x = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x)                                
        tap = loss_func(y, modell(x, training=False))    
    grad = tape.gradient(tap, x)                       
    return tf.clip_by_value(x + eps * tf.sign(grad), 0.0, 1.0).numpy()

def fgsm_malrettet(modell, x, maal, eps):
    """Målrettet FGSM: minimerer tapet mot målklassen"""
    x = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x)
        tap = loss_func(maal, modell(x, training=False))
    grad = tape.gradient(tap, x)
    return tf.clip_by_value(x - eps * tf.sign(grad), 0.0, 1.0).numpy()


def tilfeldig_stoy(x, eps):
    """Tilfeldig støy av samme størrelse som FGSM: hver piksel får ±ε med tilfeldig fortegn."""
    return np.clip(x + eps * rng.choice([-1.0, 1.0], size=x.shape), 0, 1).astype("float32")

def vis_angrep(modell, x, x_adv, antall=4):
    """Originalbilde, forstyrrelse (x_adv − x) og adversarial bilde, med prediksjon og konfidens."""
    p0, p1 = modell.predict(x[:antall], verbose=0), modell.predict(x_adv[:antall], verbose=0)
    eps = max(float(np.abs(x_adv[:antall] - x[:antall]).max()), 1e-6)
    fig, ax = plt.subplots(3, antall, figsize=(3 * antall, 9))
    for i in range(antall):
        ax[0, i].imshow(x[i, :, :, 0], cmap="gray_r"); ax[0, i].set_title(f"{klasser[p0[i].argmax()]} ({p0[i].max():.0%})")
        ax[1, i].imshow(x_adv[i, :, :, 0] - x[i, :, :, 0], cmap="RdBu", vmin=-eps, vmax=eps); ax[1, i].set_title("forstyrrelse")
        ax[2, i].imshow(x_adv[i, :, :, 0], cmap="gray_r"); ax[2, i].set_title(f"{klasser[p1[i].argmax()]} ({p1[i].max():.0%})")
    for a in ax.ravel():
        a.axis("off")
    ax[0, 0].text(-6, 14, "original", rotation=90, va="center"); ax[2, 0].text(-6, 14, "adversarial", rotation=90, va="center")
    plt.tight_layout(); plt.show()

def adversarial_trening(modell, X, y, eps=0.1, alfa=0.5, epoker=4, batch=64):
    """Adversarial training (Goodfellow m.fl. 2015): nye FGSM-eksempler mot nåværende modell i hvert steg,
    og vektene oppdateres på tapet α·J(x, y) + (1 − α)·J(x_adv, y)."""
    opt = tf.keras.optimizers.Adam(1e-3)

    @tf.function
    def treningssteg(xb, yb):
        with tf.GradientTape() as tape_x:
            tape_x.watch(xb)
            t = loss_func(yb, modell(xb, training=False))
        x_adv = tf.clip_by_value(xb + eps * tf.sign(tape_x.gradient(t, xb)), 0.0, 1.0)
        with tf.GradientTape() as tape:
            tap = alfa * loss_func(yb, modell(xb, training=True)) + (1 - alfa) * loss_func(yb, modell(x_adv, training=True))
        opt.apply_gradients(zip(tape.gradient(tap, modell.trainable_variables), modell.trainable_variables))
        return tap

    ds = tf.data.Dataset.from_tensor_slices((X, y)).shuffle(len(X), seed=0).batch(batch)
    for ep in range(epoker):
        print(f"Epoke {ep + 1}/{epoker}: tap {np.mean([float(treningssteg(xb, yb)) for xb, yb in ds]):.3f}")
    return modell



### Oppgave 1 – Bruk FGSM på modellen
- a) Lag adversarial eksempler av de fire første bildene i `X_ev` med `fgsm` og ε = 0,05, og vis dem med `vis_angrep`.
- b) Prøv også ε = 0,01 og ε = 0,1. Ved hvilken ε begynner modellen å ta feil? Kan du se forskjell på originalbildet og det adversariale bildet?

In [ ]:
eps = 0.05
x_adv = fgsm(cnn, X_ev[:4], y_ev[:4], eps)
vis_angrep(cnn, X_ev[:4], x_adv)

# TODO b): gjenta med eps = 0.01 og eps = 0.1

*Ditt svar:*

### Oppgave 2 – Hvor robuste er modellene?
- a) Beregn nøyaktigheten til MLP og CNN for ε ∈ {0, 0,01, 0,02, 0,05, 0,1, 0,2} under FGSM, og under tilfeldig støy med samme ε (`tilfeldig_stoy`). Plott resultatet.
- b) Hva er gjennomsnittlig konfidens på feilklassifiserte eksempler ved ε = 0,1?

In [ ]:
epsiloner = [0.0, 0.01, 0.02, 0.05, 0.1, 0.2]
rader = []
for e in epsiloner:
    for m in (mlp, cnn):
        x_fgsm = fgsm(m, X_ev, y_ev, e)
        rader.append({"ε": e, "modell": m.name, "angrep": "FGSM", **evaluer(m, x_fgsm, y_ev)})
        # TODO a): legg til en tilsvarende rad med angrep - bruk tilfeldig_stoy(X_ev, e))

res = pd.DataFrame(rader)
sns.lineplot(data=res, x="ε", y="nøyaktighet", hue="modell", style="angrep", markers=True)
plt.title("FGSM vs. tilfeldig støy")
plt.show()

# TODO b): vis radene for ε = 0.1, f.eks. res[res["ε"] == 0.1]

*Ditt svar:*

### Oppgave 3 – Hvilke klasser er mest sårbare?
Beregn CNN-ets nøyaktighet per klasse på opprinnelige data og etter manipulering med FGSM med ε = 0,02. Hvilke klasser er mest og minst robuste? Har du noen tanker om hvorfor det er slik?

In [ ]:
eps = 0.02
pred_ren = cnn.predict(X_ev, verbose=0).argmax(axis=1)
pred_adv = cnn.predict(fgsm(cnn, X_ev, y_ev, eps), verbose=0).argmax(axis=1)

per_klasse = pd.DataFrame(index=klasser)
per_klasse["ren"] = [(pred_ren[y_ev == k] == k).mean() for k in range(10)]
# TODO: legg til kolonnen "FGSM ε=0,02" på samme måte (bruk pred_adv) og kolonnen "fall" = ren − FGSM

per_klasse.round(3)    # TODO: sorter, f.eks. per_klasse.sort_values("FGSM ε=0,02")

*Ditt svar:*

### Oppgave 4 – Målrettet angrep
Angriperen vil nå at alle bilder skal klassifiseres som Veske (klasse 8). Blant de fohåndsdefinerte funksjonene har du `fgsm_malrettet` som egner seg til dette.
- a) Beregn andelen (av bildene som ikke allerede er vesker) som klassifiseres som Veske for ε ∈ {0,02, 0,05, 0,1}.
- b) Vis noen av bildene etter det målrettede angrepet med `vis_angrep`.


In [ ]:
MAAL = 8 # Veske
u = y_ev != MAAL # bare bilder som ikke allerede er vesker
x_u, maal = X_ev[u], np.full(u.sum(), MAAL)

for e in [0.02, 0.05, 0.1]:
    x_ett_steg = fgsm_malrettet(cnn, x_u, maal, e)
    andel_ett_steg = (cnn.predict(x_ett_steg, verbose=0).argmax(axis=1) == MAAL).mean()
    # TODO:
    print(f"ε = {e}: andel «Veske» – ett steg {andel_ett_steg:.1%}")

# TODO b): vis_angrep(cnn, x_u, ...) med bildene fra det iterative angrepet ved ε = 0.1

*Ditt svar:*

### Oppgave 5 – Overførbarhet (black-box)
Lag FGSM-eksempler med ε = 0,1 mot MLP-et og evaluer dem på CNN-et, og omvendt. Sett opp en 2 × 2-tabell. Hva betyr resultatet for en angriper som *ikke* har tilgang til modellens vekter?

In [ ]:
e = 0.1
x_fra_mlp = fgsm(mlp, X_ev, y_ev, e)          # angrep laget mot MLP-et
x_fra_cnn = fgsm(cnn, X_ev, y_ev, e)          # angrep laget mot CNN-et

tabell = pd.DataFrame(index=["evaluert på MLP", "evaluert på CNN"])
tabell["angrep laget mot MLP"] = [evaluer(mlp, x_fra_mlp, y_ev)["nøyaktighet"], evaluer(cnn, x_fra_mlp, y_ev)["nøyaktighet"]]
# TODO: fyll inn kolonnen "angrep laget mot CNN" på samme måte

tabell.round(3)

*Ditt svar:*

### Oppgave 6 – Forsvar: adversarial training
- a) Tren et nytt CNN med `adversarial_trening` (α = 0,5, ε = 0,1, 4 epoker). Det tar noen minutter på CPU.
- b) Sammenlign standard- og robust CNN: nøyaktighet under FGSM for ε ∈ {0, 0,02, 0,05, 0,1, 0,15}. Plott og lag en tabell.

In [ ]:
# a) Tren et robust CNN
robust = lag_cnn("robust_CNN")
adversarial_trening(robust, X_train, y_train, eps=0.1, alfa=0.5, epoker=4)

# b) FGSM mot begge modellene
rader = []
for e in [0.0, 0.02, 0.05, 0.1, 0.15]:
    for m in (cnn, robust):
        rader.append({"ε": e, "modell": m.name, **evaluer(m, fgsm(m, X_ev, y_ev, e), y_ev)})
res6 = pd.DataFrame(rader)
# TODO b): plott nøyaktighet mot ε for de to modellene (se oppgave 2), og vis
#          res6.pivot(index="ε", columns="modell", values="nøyaktighet")


*Ditt svar:*